# 🤖 Coding agent — write & run Python (pi-agent-core)

The capstone of notebook **100** introduced the `Agent` class. Here we turn it into a
real **coding agent**: we give it tools that actually **write files** and **execute
Python**, then ask it to build a program and run it — all confined to a sandboxed
`environment1/` workspace.

This notebook uses a **separate Azure deployment** from the env var
`AZURE_PI_TEST_DEPLOYMENT2` (point it at a capable, tool-using model).

Same stack as notebook 100: `@earendil-works/pi-agent-core`'s `Agent`, wired to our
custom Azure provider via `streamFn`. The difference is the tools — this time they
touch the filesystem and spawn a subprocess.

> **Kernel:** Deno. The Deno Jupyter kernel runs with file, read, and run permissions,
> which the tools below need (`Deno.writeTextFile`, `Deno.Command`).

## 1. Load your `.env`

In [ ]:
import { loadEnvUp } from "playground/env";
const env = await loadEnvUp();

## 2. Register the coding deployment (`AZURE_PI_TEST_DEPLOYMENT2`)

`registerAzure()` now accepts a `deployments` override, so we point it at the second
deployment instead of the default `AZURE_PI_TEST_DEPLOYMENT`.

In [ ]:
import { registerAzure } from "playground/azure";

const deployment2 = Deno.env.get("AZURE_PI_TEST_DEPLOYMENT2");
if (!deployment2) {
  throw new Error(
    "Set AZURE_PI_TEST_DEPLOYMENT2 in your .env — this coding-agent notebook uses it.",
  );
}

const { models, model } = registerAzure({ deployments: [deployment2] });
console.log(`Coding agent model: azure-openai/${deployment2}`);

## 3. A sandboxed workspace

Everything the agent creates or runs stays inside `environment1/` (next to this
notebook). We resolve every tool path against that root and **reject any path that
escapes it** (`..`), so the agent can't wander the filesystem.

In [ ]:
import { dirname, relative, resolve } from "node:path";

// The Deno Jupyter kernel's cwd is the notebook's folder, so environment1 lands here.
const SANDBOX = resolve(Deno.cwd(), "environment1");
await Deno.mkdir(SANDBOX, { recursive: true });

/** Resolve a tool-supplied relative path inside SANDBOX, refusing escapes. */
function resolveInSandbox(p: string): string {
  const abs = resolve(SANDBOX, p);
  const rel = relative(SANDBOX, abs);
  if (rel === "" || rel.startsWith("..")) {
    throw new Error(`Path "${p}" escapes the sandbox.`);
  }
  return abs;
}

console.log(`🗂️  Sandbox: ${SANDBOX}`);

## 4. The agent's tools (all sandbox-scoped)

An `AgentTool` is a `Tool` (TypeBox `parameters`) plus a `label` and an `execute`
function. The `Agent` calls `execute` automatically whenever the model emits a tool
call, then feeds the returned `content` back to the model as a tool result.

- **`write_file`** — create/overwrite a text file in the workspace.
- **`run_python`** — run a `.py` file with `python3` and capture stdout/stderr/exit code.
- **`list_files`** — list what's in the workspace.

In [ ]:
import { Type } from "@earendil-works/pi-ai";
import type { AgentTool } from "@earendil-works/pi-agent-core";

const writeFileTool: AgentTool = {
  name: "write_file",
  label: "Write File",
  description: "Create or overwrite a text file in the workspace. `path` is relative to the workspace root.",
  parameters: Type.Object({
    path: Type.String({ description: "Relative file path, e.g. hello.py" }),
    content: Type.String({ description: "Full file contents" }),
  }),
  execute: async (_id, params) => {
    const { path, content } = params as { path: string; content: string };
    const abs = resolveInSandbox(path);
    await Deno.mkdir(dirname(abs), { recursive: true });
    await Deno.writeTextFile(abs, content);
    return {
      content: [{ type: "text", text: `Wrote ${path} (${content.length} bytes).` }],
      details: { path },
    };
  },
};

const runPythonTool: AgentTool = {
  name: "run_python",
  label: "Run Python",
  description: "Run a Python file in the workspace with python3 and return stdout, stderr and the exit code.",
  parameters: Type.Object({
    file: Type.String({ description: "Relative path to the .py file to run" }),
  }),
  execute: async (_id, params, signal) => {
    const { file } = params as { file: string };
    const abs = resolveInSandbox(file);
    const { code, stdout, stderr } = await new Deno.Command("python3", {
      args: [abs],
      cwd: SANDBOX,
      stdout: "piped",
      stderr: "piped",
      signal,
    }).output();
    const out = new TextDecoder().decode(stdout);
    const err = new TextDecoder().decode(stderr);
    const text = `exit code: ${code}\n--- stdout ---\n${out}` + (err ? `--- stderr ---\n${err}` : "");
    return {
      content: [{ type: "text", text }],
      details: { code, stdout: out, stderr: err },
      isError: code !== 0,
    };
  },
};

const listFilesTool: AgentTool = {
  name: "list_files",
  label: "List Files",
  description: "List files in the workspace.",
  parameters: Type.Object({}),
  execute: async () => {
    const names: string[] = [];
    for await (const e of Deno.readDir(SANDBOX)) names.push(e.isDirectory ? `${e.name}/` : e.name);
    return { content: [{ type: "text", text: names.length ? names.join("\n") : "(empty)" }], details: { names } };
  },
};

const tools = [writeFileTool, runPythonTool, listFilesTool];
console.log(`Defined ${tools.length} sandboxed tools: ${tools.map((t) => t.name).join(", ")}`);

## 5. Build the agent

We wire the `Agent` to our custom Azure `models` collection with a one-line `streamFn`
(exactly as in notebook 100), give it the sandbox tools, and a system prompt that tells
it to write code, run it, and report the real output.

In [ ]:
import { Agent, type StreamFn } from "@earendil-works/pi-agent-core";

const streamFn: StreamFn = (m, ctx, opts) => models.streamSimple(m, ctx, opts);

const agent = new Agent({
  streamFn,
  initialState: {
    model,
    systemPrompt:
      "You are a Python coding agent working in a sandboxed workspace. " +
      "Use write_file to create files and run_python to execute them. " +
      "All paths are relative to the workspace root. " +
      "When asked to build a program: write the file, run it with run_python, " +
      "then report the program's exact output. Keep the code minimal.",
    tools,
  },
});

// Observe the loop: stream the assistant's text and print each tool result.
const enc = new TextEncoder();
agent.subscribe((event) => {
  switch (event.type) {
    case "message_update":
      if (event.assistantMessageEvent.type === "text_delta") {
        Deno.stdout.writeSync(enc.encode(event.assistantMessageEvent.delta));
      }
      break;
    case "turn_end":
      for (const tr of event.toolResults) {
        const text = tr.content.map((c) => (c.type === "text" ? c.text : "")).join("");
        console.log(`\n🔧 ${tr.toolName}:\n${text}`);
      }
      break;
    case "agent_end":
      console.log("\n\n✅ Agent finished.");
      break;
  }
});

console.log("Agent ready.");

## 6. Run it — write & execute a Python hello world

One `prompt()` kicks off the whole loop: the agent decides to call `write_file`, then
`run_python`, and reports the output — no manual loop on our side. `waitForIdle()`
resolves once the run (and all event listeners) have settled.

In [ ]:
await agent.prompt(
  "Create a Python program named `hello.py` that prints exactly: Hello, World! " +
  "Then run it with run_python and tell me its output.",
);
await agent.waitForIdle();

## 7. Proof: show and re-run the generated file

The agent already ran the program (see the `run_python` result above). To make the
result unambiguous, we read the file it wrote and execute it ourselves.

In [ ]:
const helloPath = resolve(SANDBOX, "hello.py");
console.log("📄 environment1/hello.py:\n");
console.log(await Deno.readTextFile(helloPath));

const proof = await new Deno.Command("python3", {
  args: [helloPath],
  cwd: SANDBOX,
  stdout: "piped",
  stderr: "piped",
}).output();
console.log("\n▶️  python3 hello.py →");
console.log(new TextDecoder().decode(proof.stdout));

## ✅ What just happened

1. `registerAzure({ deployments: [AZURE_PI_TEST_DEPLOYMENT2] })` registered a dedicated
   coding deployment.
2. We defined three **sandboxed** `AgentTool`s (`write_file`, `run_python`,
   `list_files`) that refuse to touch anything outside `environment1/`.
3. An `Agent` (from `@earendil-works/pi-agent-core`) wired to Azure via `streamFn` ran
   the full loop: it wrote `hello.py`, executed it, and reported the output — driven by
   a single `prompt()`.

This is the manual tool loop of notebook 040, but the framework owns the loop and the
tools do real work (filesystem + subprocess) inside a safe workspace.

Next: **120 — the same idea via `@earendil-works/pi-coding-agent`'s `createAgentSession`**,
the higher-level coding-harness SDK with built-in tools.